# Hero pick rate and win rate (ranked All Pick)

Reads the dbt mart `marts.hero_pick_win_rate`, built from the Steam match summaries (ranked lobby, All Pick,
15+ minutes, 10 human players, known winner). Set `DUCKDB_PATH` to read a local dev DuckDB file; otherwise the
warehouse Postgres is read through the `WAREHOUSE_*` environment variables. Install dependencies with
`python -m pip install -r notebooks/requirements.txt`.

In [ ]:
import os

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

MART = "marts.hero_pick_win_rate"
TOP_N = 15
MIN_PICKS = 3  # heroes below this many picks are left out of the win-rate chart (too few games to mean much)


def load_mart() -> pd.DataFrame:
    """The whole mart as a DataFrame: DuckDB when DUCKDB_PATH is set, else the warehouse Postgres."""
    query = f"SELECT * FROM {MART}"
    if os.environ.get("DUCKDB_PATH"):
        con = duckdb.connect(os.environ["DUCKDB_PATH"], read_only=True)
        try:
            return con.execute(query).df()
        finally:
            con.close()
    import psycopg2

    conn = psycopg2.connect(
        host=os.environ.get("WAREHOUSE_HOST", "localhost"), port=os.environ.get("WAREHOUSE_PORT", "5432"),
        user=os.environ.get("WAREHOUSE_USER", "warehouse"), password=os.environ["WAREHOUSE_PASSWORD"],
        dbname=os.environ.get("WAREHOUSE_DB", "warehouse"))
    try:
        with conn.cursor() as cur:
            cur.execute(query)
            return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
    finally:
        conn.close()


mart = load_mart()
ranked_matches = int(mart["ranked_matches"].iloc[0]) if len(mart) else 0
mart["name"] = mart["localized_name"].fillna(mart["hero_id"].astype(str))
print(f"{len(mart)} heroes, {ranked_matches} ranked matches")

## Most picked heroes

In [ ]:
top = mart.sort_values("pick_rate", ascending=False).head(TOP_N).iloc[::-1]  # reversed so the top hero is drawn at the top
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top["name"], top["pick_rate"] * 100)
ax.set_xlabel("Pick rate (% of ranked matches)")
ax.set_ylabel("Hero")
ax.set_title(f"Top {TOP_N} heroes by pick rate ({ranked_matches} ranked matches)")
fig.tight_layout()
plt.show()

## Win rate of heroes with enough picks

In [ ]:
eligible = mart[mart["picks"] >= MIN_PICKS].sort_values("win_rate").tail(TOP_N * 2)
fig, ax = plt.subplots(figsize=(8, max(3, 0.3 * len(eligible) + 1)))
ax.barh(eligible["name"], eligible["win_rate"] * 100)
ax.axvline(50, color="grey", linestyle="--", linewidth=1)
ax.set_xlabel("Win rate (% of picks)")
ax.set_ylabel("Hero")
ax.set_title(f"Win rate of heroes with at least {MIN_PICKS} picks ({ranked_matches} ranked matches)")
fig.tight_layout()
plt.show()